In [1]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
import sys
import tqdm
import pathlib
from pathlib import Path
import io
import contextlib
import gaps_online as go
import go_pybindings as gop
import re
from glob import glob
import os

[gaps-db] Unable to import vtk, plotting options might be limited!
Can't load CXX API! No module named 'gaps_tof'
Unable to load CXX API! No module named 'gaps_tof'
Can't find charmingbeauty for nice looking plots! No module named 'charmingbeauty'


In [2]:
dataset = pathlib.Path(f'/home/gaps/csbf-data/136')
files = [f for f in sorted(dataset.glob('*.tof.gaps'))]

In [3]:
paddle_map = {}
with open(f'/home/gaps/userspace/grace/analysis/resources/channel_mapping.csv') as in_file:
    variables = next(in_file).strip().split(',')
    next(in_file)
    for line in in_file:
        row = line.strip().split(',')
        paddle_id = int(row[0])
        paddle_map[paddle_id] = {'a':{'rb':0,'ch':0},'b':{'rb':0,'ch':0}}
        rb, ch = [int(d) for d in row[9].split('-')]
        paddle_map[paddle_id]['a']['rb'] = rb
        paddle_map[paddle_id]['a']['ch'] = ch - 1

        row = next(in_file).strip().split(',')
        rb, ch = [int(d) for d in row[9].split('-')]
        paddle_map[paddle_id]['b']['rb'] = rb
        paddle_map[paddle_id]['b']['ch'] = ch - 1

In [4]:
pattern = re.compile(r'RB(\d+)_\d{6}_\d{6}UTC\.cali\.tof\.gaps')
calibrations = glob('/home/gaps/csbf-data/calib/240806_030946UTC/*.cali.tof.gaps')

calib = {}

for fname in calibrations:
    match = pattern.search(fname)
    if match:
        rbid = match.group(1)
        cali = gop.events.RBCalibration()
        cali.from_file(fname)  # Modify the instance
        calib[int(rbid)] = cali      # Store the modified instance
    else:
        print("No match found for:", fname)

In [6]:
mangling_from_status = 0
mangling_from_wv = 0

for f in files[:1]:
        reader = go.rust_api.io.TofPacketReader(str(f), filter=go.rust_api.io.PacketType.TofEvent)
        settings = go.liftof.LiftofSettings()
        settings = settings.from_file('/home/gaps/csbf-data/134/run134.toml')

        n_packets = 0
        for pack in reader:
            n_packets += 1

        reader.rewind()
    
        for pack in tqdm.tqdm(reader, total=n_packets, file=sys.stdout, position=0):
            ev = go.rust_api.events.TofEvent()
            new_ev = go.liftof.waveform_analysis(ev, settings)

            try:
                ev.from_tofpacket(pack)
                status = ev.mastertriggerevent.status
                #mangling.append(int(status))
                if int(status) == 16:
                    mangling_from_status += 1
        
            except Exception as e:
                print(f"Error: {e}")
                pass
                continue
            
            for x in range(len(new_ev.hits)):
                try: 
                    paddle = int(new_ev.hits[x].paddle_id)

                    if new_ev.hits[x].charge_a == 0 or new_ev.hits[x].charge_b == 0:
                        continue


                    rb = paddle_map[paddle]['a']['rb']
                    ch = paddle_map[paddle]['a']['ch']
                    if ch == 8: continue
                    for waveform in new_ev.waveforms:
                        if waveform.rb_id == rb and waveform.rb_channel == ch:
                            waveform.calibrate(calib[rb])
                            waveform.apply_spike_filter()
                            if min(waveform.voltages < -200): 
                                mangling_from_wv += 1
                                break
                    
                    rb = paddle_map[paddle]['b']['rb']
                    ch = paddle_map[paddle]['b']['ch']
                    if ch == 8: continue
                    for waveform in new_ev.waveforms:
                        if waveform.rb_id == rb and waveform.rb_channel == ch:
                            waveform.calibrate(calib[rb])
                            waveform.apply_spike_filter()
                            if min(waveform.voltages < -200): 
                                mangling_from_wv += 1
                                break
                except Exception as e:
                    print(f"Error at hit {x}: {e}")
                    continue

  2%|██▉                                                                                                                           | 674/29221 [00:10<07:23, 64.44it/s]


KeyboardInterrupt: 

In [ ]:
print(mangling_from_status, mangling_from_wv)

In [ ]:
min(mangling)

In [ ]:
status == 'Unknown'

In [ ]:
type(status.AnyDataMangling)

In [ ]:
status

In [ ]:
if status.Perfect:
    print('agfasf')

In [ ]:
int(status)